In [4]:
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset,DataLoader
import pickle

In [5]:
###数据处理
#句子 -> 索引序列； 按最大长度填充；  处理成batchsize大小； 包装成 tensor
class MyDataset(Dataset):
    def __init__(self,en_data,ch_data,en_word_2_index,ch_word_2_index):
        self.en_data = en_data
        self.ch_data = ch_data
        self.en_word_2_index = en_word_2_index   #词语索引
        self.ch_word_2_index = ch_word_2_index

    #获取中英文单词的索引
    def __getitem__(self,index):
        en = self.en_data[index] #读一个句子
        ch = self.ch_data[index]

        en_index = [self.en_word_2_index[i] for i in en]
        ch_index = [self.ch_word_2_index[i] for i in ch]

        return en_index,ch_index

    #将索引表示的句子处理成batch数据,并按最长句子长度填充,封装成tensor
    def batch_data_process(self,batch_data): #一个batch的数据
        global device
        en_index , ch_index = [],[]
        en_len , ch_len = [],[]

        for en,ch in batch_data: 
            en_index.append(en)   ##英文句子（索引）
            ch_index.append(ch)
            en_len.append(len(en))  #英文句子长度
            ch_len.append(len(ch))

        max_en_len = max(en_len)  #最大英文句子长度
        max_ch_len = max(ch_len)

        en_index = [ i + [self.en_word_2_index["<PAD>"]] * (max_en_len - len(i))   for i in en_index]  #<PAD>填充
        ch_index = [[self.ch_word_2_index["<BOS>"]]+ i + [self.ch_word_2_index["<EOS>"]] + [self.ch_word_2_index["<PAD>"]] * (max_ch_len - len(i))   for i in ch_index]

        en_index = torch.tensor(en_index,device = device)
        ch_index = torch.tensor(ch_index,device = device)
        return en_index,ch_index


    def __len__(self):   #双语条数匹配检测
        assert len(self.en_data) == len(self.ch_data)
        return len(self.ch_data)

In [6]:
###读数据
def get_data(file = "translate.csv",nums = None):
    all_data = pd.read_csv(file)
    en_data = list(all_data["english"])   #平行语料英文
    ch_data = list(all_data["chinese"])

    if nums == None:
        return en_data,ch_data
    else:
        return en_data[:nums],ch_data[:nums]

t1,t2=get_data()
print(t1[:5],t2[:5])

['Hi.', 'Hi.', 'Run.', 'Wait!', 'Wait!'] ['嗨。', '你好。', '你用跑的。', '等等！', '等一下！']


In [7]:
###编码器,解码器
class Encoder(nn.Module):
    def __init__(self,encoder_embedding_num,encoder_hidden_num,en_vocab_len):
        super().__init__()
        # 填空1: 创建英文词嵌入层
        # self.embedding = nn.______(______, ______) #英文词典长度，嵌入维度
        self.embedding = nn.Embedding(en_vocab_len, encoder_embedding_num) #英文词典长度，嵌入维度

        # 填空2: 创建LSTM编码器
        # self.lstm = nn.LSTM(______, ______, batch_first=True)
        self.lstm = nn.LSTM(encoder_embedding_num, encoder_hidden_num, batch_first=True)


    def forward(self,en_index):
        # 填空3: 将英文索引转换为词向量
        # en_embedding = ______(______)
        en_embedding = self.embedding(en_index)

        # 填空4: LSTM编码，返回隐藏状态
        # _, encoder_hidden = ______(______)  #输出：最终隐藏向量，每一步的隐层向量
        _, encoder_hidden = self.lstm(en_embedding)  #输出：最终隐藏向量，每一步的隐层向量

        return encoder_hidden

class Decoder(nn.Module):
    def __init__(self,decoder_embedding_num,decoder_hidden_num,ch_vocab_len):
        super().__init__()
        # 填空5: 创建中文词嵌入层
        # self.embedding = nn.______(______, ______)
        self.embedding = nn.Embedding(ch_vocab_len, decoder_embedding_num)

        # 填空6: 创建LSTM解码器
        # self.lstm = nn.LSTM(______, ______, batch_first=True)
        self.lstm = nn.LSTM(decoder_embedding_num, decoder_hidden_num, batch_first=True)


    def forward(self,decoder_input, hidden):
        # 填空7: 将中文索引转换为词向量
        # embedding = ______(______)
        embedding = self.embedding(decoder_input)

        # 填空8: LSTM解码，使用编码器隐藏状态
        # decoder_output, decoder_hidden = ______(______, ______) #输出：解码器输出和新的隐藏状态
        decoder_output, decoder_hidden = self.lstm(embedding, hidden) #输出：解码器输出和新的隐藏状态

        return decoder_output,decoder_hidden

In [8]:
###Seq2Seq
class Seq2Seq(nn.Module):
    def __init__(self,encoder_embedding_num,encoder_hidden_num,en_vocab_len,decoder_embedding_num,decoder_hidden_num,ch_vocab_len):
        super().__init__()
        # 填空9: 初始化编码器
        # self.encoder = ______(______, ______, ______)
        self.encoder = Encoder(encoder_embedding_num, encoder_hidden_num, en_vocab_len)

        # 填空10: 初始化解码器
        # self.decoder = ______(______, ______, ______)
        self.decoder = Decoder(decoder_embedding_num, decoder_hidden_num, ch_vocab_len)

        # 填空11: 创建输出分类器
        # self.classifier = nn.______(______, ______)    # 将隐藏状态映射到词汇表
        self.classifier = nn.Linear(decoder_hidden_num, ch_vocab_len)    # 将隐藏状态映射到词汇表

        # 定义损失函数
        self.cross_loss = nn.CrossEntropyLoss()



    def forward(self,en_index,ch_index):
        # 填空12: 准备解码器输入（教师强制训练）
        # decoder_input = ch_index[:, ______] #去掉句子的最后一个token
        decoder_input = ch_index[:, :-1]

        # 填空13: 准备训练标签
        # label = ch_index[:, ______] #去掉句子的第一个token
        label = ch_index[:, 1:]
        
        # 填空14: 编码英文序列
        # encoder_hidden = ______(______)  #获得编码器隐藏状态
        encoder_hidden = self.encoder(en_index)  #获得编码器隐藏状态
        
        # 填空15: 解码生成中文序列
        # decoder_output, _ = ______(______, ______)   #使用编码器隐藏状态初始化解码器
        decoder_output, _ = self.decoder(decoder_input, encoder_hidden)
        
        pre = self.classifier(decoder_output)  # 形状: (batch_size, seq_len, ch_vocab_len)
        
        loss = self.cross_loss(pre.reshape(-1,pre.shape[-1]),label.reshape(-1))  #计算损失
        
        return loss

In [9]:
def translate(sentence):
    global en_word_2_index,model,device,ch_word_2_index,ch_index_2_word
    en_index = torch.tensor([[en_word_2_index[i] for i in sentence]],device=device) #输入的英文转换为索引

    result = []
    encoder_hidden = model.encoder(en_index)  #英文编码
    decoder_input = torch.tensor([[ch_word_2_index["<BOS>"]]],device=device)  #译文的开头<BOS>=>转为tensor

    decoder_hidden = encoder_hidden  #编码器输出作为解码器的第一个隐层的输入
    while True:
        decoder_output,decoder_hidden = model.decoder(decoder_input,decoder_hidden)  #从<BOS>开始,逐步输出下一个预测向量
        pre = model.classifier(decoder_output)  # h->y

        w_index = int(torch.argmax(pre,dim=-1))   #采用贪心解码，argmax取y中概率最大的词
        word = ch_index_2_word[w_index]

        if word == "<EOS>" or len(result) > 50:  #限制输出为50
            break

        result.append(word)
        decoder_input = torch.tensor([[w_index]],device=device)

    print("译文: ","".join(result))

In [11]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"

with open("ch.vec","rb") as f1:   #中文词典
    _, ch_word_2_index,ch_index_2_word = pickle.load(f1)
    
with open("en.vec","rb") as f2:
    _, en_word_2_index, en_index_2_word = pickle.load(f2)

print(list(ch_word_2_index)[:5])
print(list(en_word_2_index)[:5])

['。', '我', '的', '了', '你']
[' ', 'e', 'o', 't', 'a']


In [12]:
ch_vocab_len = len(ch_word_2_index)  #中文词典长度,以字为粒度
en_vocab_len = len(en_word_2_index)  #英文词典长度,以字母为粒度
print(ch_vocab_len, en_vocab_len)

3589 77


In [13]:
ch_word_2_index.update({"<PAD>":ch_vocab_len,"<BOS>":ch_vocab_len + 1 , "<EOS>":ch_vocab_len+2})  #中文字典末尾增加3个单词
en_word_2_index.update({"<PAD>":en_vocab_len}) #英文字典增加1个


ch_index_2_word += ["<PAD>","<BOS>","<EOS>"]  ##增加3个单词
en_index_2_word += ["<PAD>"]

ch_vocab_len += 3
en_vocab_len = len(en_word_2_index)

In [14]:
en_data,ch_data = get_data(nums=200)  #
encoder_embedding_num = 50  #word embedding维度
encoder_hidden_num = 100  #隐层维度
decoder_embedding_num = 107
decoder_hidden_num = 100

batch_size = 2
epoch = 40
lr = 0.001

In [15]:
dataset = MyDataset(en_data,ch_data,en_word_2_index,ch_word_2_index)  #读数据集
dataloader = DataLoader(dataset,batch_size,shuffle=False,collate_fn = dataset.batch_data_process)

print(list(dataset)[:5]) #双语句子的索引表示
print(list(dataloader)[:5]) #双语句子索引的tensor，batch包装

[([29, 6, 12], [2085, 0]), ([29, 6, 12], [4, 33, 0]), ([57, 13, 7, 12], [4, 91, 415, 2, 0]), ([30, 4, 6, 3, 51], [205, 205, 187]), ([30, 4, 6, 3, 51], [205, 9, 43, 187])]
[(tensor([[29,  6, 12],
        [29,  6, 12]], device='cuda:0'), tensor([[3590, 2085,    0, 3591, 3589],
        [3590,    4,   33,    0, 3591]], device='cuda:0')), (tensor([[57, 13,  7, 12, 77],
        [30,  4,  6,  3, 51]], device='cuda:0'), tensor([[3590,    4,   91,  415,    2,    0, 3591],
        [3590,  205,  205,  187, 3591, 3589, 3589]], device='cuda:0')), (tensor([[30,  4,  6,  3, 51, 77],
        [45,  1, 17,  6,  7, 12]], device='cuda:0'), tensor([[3590,  205,    9,   43,  187, 3591],
        [3590,  124,  219,  187, 3591, 3589]], device='cuda:0')), (tensor([[45,  1, 17,  6,  7, 77],
        [29,  1, 10, 10,  2, 51]], device='cuda:0'), tensor([[3590,  124,  219, 3591, 3589],
        [3590,    4,   33,    0, 3591]], device='cuda:0')), (tensor([[21,  0,  3,  9, 15, 12],
        [21,  0, 16,  2,  7, 51]], de

In [16]:
model = Seq2Seq(encoder_embedding_num,encoder_hidden_num,en_vocab_len,decoder_embedding_num,decoder_hidden_num,ch_vocab_len)
model = model.to(device)

opt = torch.optim.Adam(model.parameters(),lr = lr)

for e in range(epoch):
    for en_index,ch_index  in dataloader:  #batchsize 个句子对
        loss = model(en_index,ch_index)
        loss.backward()
        opt.step()
        opt.zero_grad()

    print(f"loss:{loss:.3f}")

loss:4.128
loss:3.464
loss:3.138
loss:2.895
loss:2.690
loss:2.504
loss:2.310
loss:2.061
loss:1.850
loss:1.701
loss:1.480
loss:1.288
loss:1.072
loss:0.899
loss:0.756
loss:0.636
loss:0.505
loss:0.426
loss:0.350
loss:0.289
loss:0.236
loss:0.197
loss:0.173
loss:0.146
loss:0.128
loss:0.110
loss:0.098
loss:0.088
loss:0.080
loss:0.070
loss:0.064
loss:0.057
loss:0.051
loss:0.045
loss:0.042
loss:0.036
loss:0.036
loss:0.031
loss:0.028
loss:0.026


In [17]:
print("\n--- 开始测试翻译 ---")
test_sentences = [
        "I am fine.",
        "Be friendly.",
        "I am sick.",
        "It doesn't matter."
]

with torch.no_grad(): # 关闭梯度计算，加快速度
    for sentence in test_sentences:
        translate(sentence)


--- 开始测试翻译 ---
译文:  我很好。
译文:  友善点。
译文:  我生病了。
译文:  是是个。
